In [1]:
# ======================================================================
# DUBLIN PLANNING (processed): Dublin councils only + location columns
# ======================================================================
# Input:  dataSet\processed_dataset\planning_dataset\planning_2021_onwards.csv
# Output: just_dublin_dataset\just_dublin_processed_dataset\planning\planning_dublin.csv
#
# Dublin has 4 councils: Dublin City, Fingal, South Dublin and
# Dun Laoghaire-Rathdown. The council is the local_authority.
# The postal district is read from the development address.
#
# Needs 00_dublin_location_lookup.ipynb to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

planning_file = base + r'\dataSet\processed_dataset\planning_dataset\planning_2021_onwards.csv'
dublin_processed = base + r'\just_dublin_dataset\just_dublin_processed_dataset'
save_folder = dublin_processed + r'\planning'

os.makedirs(save_folder, exist_ok=True)

In [2]:
planning = pd.read_csv(planning_file, low_memory=False)
print('All of Ireland:', planning.shape)

All of Ireland: (225141, 36)


In [3]:
# Keeping only the 4 Dublin councils

planning = planning[planning['county'] == 'Dublin']

print('Dublin only:', planning.shape)
print(planning['PlanningAuthority'].value_counts())

Dublin only: (43351, 36)
PlanningAuthority
Dublin City Council                      18314
Dun Laoghaire Rathdown County Council     9799
Fingal County Council                     8901
South Dublin County Council               6337
Name: count, dtype: int64


In [4]:
# local_authority = the council that handled the application

planning['local_authority'] = planning['PlanningAuthority'].str.strip()
planning['local_authority'].value_counts()

local_authority
Dublin City Council                      18314
Dun Laoghaire Rathdown County Council     9799
Fingal County Council                     8901
South Dublin County Council               6337
Name: count, dtype: int64

In [5]:
# Looking at some addresses to see how districts are written

planning['DevelopmentAddress'].sample(15, random_state=1)

167639                                                                                 3, Ballyroan Crescent, Dublin 16, D16 N2Y3
69912     Former Des 'Kelly Site', a site of 0.262 Ha. bounded by 364 - 374 North Circular Road, Royal Canal Bank and 168, 169...
81736                                                                             41 Rathbeale Road, Swords, Co. Dublin, K67 D899
168445                                                                Units 3B & 4, Lidl Complex, Main Road, Tallaght, Dublin 24.
85409                                                           Lisieux, Rahillion, Portrane Road, Donabate, Co. Dublin, K36 NY17
62381                                                                               24 Kenilworth Square North, Rathgar, Dublin 6
56173                                                                                      66, Lally Road, Ballyfermot, Dublin 10
53797                                                                                     

In [6]:
# Finding the district in the address.
# The pattern looks for the word DUBLIN followed by a number, e.g.
# 'Crumlin, Dublin 12' -> '12', 'Harolds Cross, Dublin 6w' -> '6W'
#   DUBLIN   the word
#   \s*      any spaces
#   (\d+W?)  the number, maybe followed by W (for 6W)

address = planning['DevelopmentAddress'].fillna('').str.upper()
found = address.str.extract(r'DUBLIN\s*(\d+W?)')[0]

found.value_counts().head(30)

0
4     2019
14    1844
9     1826
15    1799
18    1788
6     1735
3     1704
12    1653
5     1626
7     1545
8     1418
2     1397
16    1305
24    1198
11    1160
6W     989
1      952
13     872
22     779
17     342
10     279
20     275
09       8
08       7
04       4
06       3
03       2
01       1
07       1
Name: count, dtype: int64

In [7]:
# Turning the number into a district name ('12' -> 'Dublin 12').
# Some councils write '09' or '04', so the leading 0 is removed.

def number_to_district(number):
    if pd.isnull(number):
        return np.nan
    w = ''
    if number.endswith('W'):
        w = 'W'
        number = number[:-1]
    return 'Dublin ' + str(int(number)) + w


planning['dublin_district'] = found.apply(number_to_district)
planning['dublin_district'].value_counts()

dublin_district
Dublin 4     2023
Dublin 14    1844
Dublin 9     1834
Dublin 15    1799
Dublin 18    1788
Dublin 6     1738
Dublin 3     1706
Dublin 12    1653
Dublin 5     1626
Dublin 7     1546
Dublin 8     1425
Dublin 2     1397
Dublin 16    1305
Dublin 24    1198
Dublin 11    1160
Dublin 6W     989
Dublin 1      953
Dublin 13     872
Dublin 22     779
Dublin 17     342
Dublin 10     279
Dublin 20     275
Name: count, dtype: int64

In [8]:
# Some addresses don't say 'Dublin 12' but include an Eircode instead
# (e.g. 'D16 N2Y3'). Using the Eircode area for those.

lookup = pd.read_csv(dublin_processed + r'\dublin_location_lookup.csv')
# (the 'no district' row has no Eircode, so it is left out of this map;
#  otherwise a missing Eircode would wrongly match it)
with_code = lookup[lookup['eircode_area'].notnull()]
eircode_map = dict(zip(with_code['eircode_area'], with_code['dublin_district']))

eircode_found = address.str.extract(r'\b(D\d\d|D6W)\b')[0]
from_eircode = eircode_found.map(eircode_map)

still_missing = planning['dublin_district'].isnull()
print('Filled from Eircode:', (still_missing & from_eircode.notnull()).sum())

planning.loc[still_missing, 'dublin_district'] = from_eircode[still_missing]

Filled from Eircode: 1212


In [9]:
# Anything that is not one of the real districts (e.g. a typo like
# 'Dublin 19') or has no district at all becomes 'Dublin (no district)'.

district_list = list(lookup['dublin_district'])

not_real = planning['dublin_district'].notnull() & ~planning['dublin_district'].isin(district_list)
print('Not a real district:', not_real.sum())
print(planning.loc[not_real, 'dublin_district'].value_counts())

planning.loc[~planning['dublin_district'].isin(district_list), 'dublin_district'] = 'Dublin (no district)'

Not a real district: 0
Series([], Name: count, dtype: int64)


In [10]:
# How many applications got a district?

has_district = (planning['dublin_district'] != 'Dublin (no district)').mean() * 100
print('Applications with a postal district:', round(has_district, 1), '%')
print()
print(pd.crosstab(planning['dublin_district'], planning['local_authority']))

Applications with a postal district: 68.6 %

local_authority       Dublin City Council  \
dublin_district                             
Dublin (no district)                  223   
Dublin 1                              941   
Dublin 10                             269   
Dublin 11                            1013   
Dublin 12                            1152   
Dublin 13                             329   
Dublin 14                               2   
Dublin 15                              54   
Dublin 16                               0   
Dublin 17                             308   
Dublin 18                               1   
Dublin 2                             1412   
Dublin 20                              94   
Dublin 22                               1   
Dublin 24                               1   
Dublin 3                             1723   
Dublin 4                             1996   
Dublin 5                             1631   
Dublin 6                             1762   
Dublin 6W 

In [11]:
# Adding district_id and the higher levels from the lookup

planning = planning.merge(lookup[['dublin_district', 'district_id', 'county_id',
                                  'province', 'nation']],
                          on='dublin_district', how='left')

print('Rows without a district id:', planning['district_id'].isnull().sum())   # should be 0
planning[['PlanningAuthority', 'DevelopmentAddress', 'dublin_district', 'district_id',
          'local_authority', 'county', 'province', 'nation']].head()

Rows without a district id: 0


,PlanningAuthority,DevelopmentAddress,dublin_district,district_id,local_authority,county,province,nation
0,Dublin City Council,"146 Baggot Street Lower, Dublin 2, D02 T990",Dublin 2,2,Dublin City Council,Dublin,Leinster,Ireland
1,Dublin City Council,"98-100 South Circular Road, Dublin 8, D08 V5YA",Dublin 8,9,Dublin City Council,Dublin,Leinster,Ireland
2,Dublin City Council,"7, Church Park Court, Dublin 6w, D6W DW65",Dublin 6W,7,Dublin City Council,Dublin,Leinster,Ireland
3,Dublin City Council,"2 Rathborne Chase, Boulevard, Ashtown, Dublin 15, D15 NW6V",Dublin 15,16,Dublin City Council,Dublin,Leinster,Ireland
4,Dublin City Council,"151 Pearse Street, Dublin 2, D02 C898",Dublin 2,2,Dublin City Council,Dublin,Leinster,Ireland


In [12]:
planning.to_csv(save_folder + r'\planning_dublin.csv', index=False)
print('Saved', planning.shape, 'to', save_folder + r'\planning_dublin.csv')

Saved (43351, 42) to C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\just_dublin_dataset\just_dublin_processed_dataset\planning\planning_dublin.csv
